[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/05_optimization/04.5_chunked_prefill/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io)

# Lab 4.5: Chunked Prefill Simulation

This lab simulates how chunked prefill prevents decode stalls during long prefill operations. We model a continuous batching scheduler with and without chunking to visualize the ITL impact.

In [ ]:
# Install required plotting and numerical libraries
import subprocess
import sys
# Use subprocess to ensure matplotlib and numpy are present
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "numpy"])

In [ ]:
# Import numerical library for array operations and statistics
import numpy as np
# Import plotting library for latency visualizations
import matplotlib.pyplot as plt

# ============================================================
# PARAMETERS: Modify these to explore different scenarios
# ============================================================
# Number of requests currently in the decode phase
NUM_ACTIVE_REQUESTS = 64
# Time in ms to process one decode token per request
DECODE_TIME_PER_TOKEN_MS = 0.2
# Length of the incoming long prompt (tokens)
PREFILL_TOKENS = 8192
# Time in ms to process one prefill token
PREFILL_TIME_PER_TOKEN_MS = 0.05
# Maximum tokens the scheduler allows per iteration
TOKEN_BUDGET = 4096
# Total iterations to simulate in the timeline
NUM_ITERATIONS = 200
# Iteration index when the long prompt arrives
LONG_PREFILL_ARRIVES_AT = 50

In [ ]:
def simulate_no_chunking(
    num_active: int,
    prefill_tokens: int,
    prefill_time_per_token: float,
    decode_time_per_token: float,
    num_iterations: int,
    arrival_iter: int,
) -> np.ndarray:
    # Simulate ITL without chunked prefill
    # When a long prefill arrives it blocks ALL decode for its full duration
    # Returns per-iteration decode latency (ms) for active requests

    # Total time the prefill takes to complete
    prefill_duration_ms = prefill_tokens * prefill_time_per_token
    # Normal decode iteration time (all active requests get 1 token)
    normal_iter_time_ms = num_active * decode_time_per_token
    # How many normal iterations are blocked by the prefill
    blocked_iters = int(np.ceil(prefill_duration_ms / normal_iter_time_ms))

    # Initialize all iterations with normal decode time
    itl = np.full(num_iterations, normal_iter_time_ms)
    # During prefill block: existing requests see elevated ITL
    # The GPU is busy with prefill so decode gets delayed
    itl[arrival_iter:arrival_iter + blocked_iters] = (
        prefill_duration_ms / blocked_iters + normal_iter_time_ms
    )
    # Return the ITL timeline array
    return itl


def simulate_chunked_prefill(
    num_active: int,
    prefill_tokens: int,
    prefill_time_per_token: float,
    decode_time_per_token: float,
    token_budget: int,
    num_iterations: int,
    arrival_iter: int,
) -> np.ndarray:
    # Simulate ITL with chunked prefill enabled
    # Prefill is split into chunks that share iteration time with decode
    # Returns per-iteration decode latency (ms) for active requests

    # Available prefill tokens per iteration = budget minus decode slots
    chunk_size = token_budget - num_active
    # Number of iterations needed to complete the full prefill
    num_chunks = int(np.ceil(prefill_tokens / chunk_size))

    # Time for decode-only iteration (baseline)
    normal_iter_time_ms = num_active * decode_time_per_token
    # Additional time contributed by the prefill chunk
    chunk_prefill_time_ms = chunk_size * prefill_time_per_token
    # Total iteration time when a chunk is being processed
    chunked_iter_time_ms = normal_iter_time_ms + chunk_prefill_time_ms

    # Initialize all iterations with normal decode time
    itl = np.full(num_iterations, normal_iter_time_ms)
    # During chunked prefill: ITL increases slightly (shared compute)
    # But decode still happens every iteration (no stall)
    itl[arrival_iter:arrival_iter + num_chunks] = chunked_iter_time_ms
    # Return the ITL timeline array
    return itl

In [ ]:
# Run the simulation without chunked prefill
itl_no_chunk = simulate_no_chunking(
    NUM_ACTIVE_REQUESTS, PREFILL_TOKENS, PREFILL_TIME_PER_TOKEN_MS,
    DECODE_TIME_PER_TOKEN_MS, NUM_ITERATIONS, LONG_PREFILL_ARRIVES_AT
)

# Run the simulation with chunked prefill enabled
itl_chunked = simulate_chunked_prefill(
    NUM_ACTIVE_REQUESTS, PREFILL_TOKENS, PREFILL_TIME_PER_TOKEN_MS,
    DECODE_TIME_PER_TOKEN_MS, TOKEN_BUDGET, NUM_ITERATIONS, LONG_PREFILL_ARRIVES_AT
)

# Print comparison table of latency percentiles
print("=== ITL Statistics (ms) ===")
print(f"{'Metric':<12} {'No Chunking':>12} {'Chunked':>12}")
# P50 = median latency experienced by active requests
print(f"{'P50':<12} {np.percentile(itl_no_chunk, 50):>12.1f} {np.percentile(itl_chunked, 50):>12.1f}")
# P95 = 95th percentile latency
print(f"{'P95':<12} {np.percentile(itl_no_chunk, 95):>12.1f} {np.percentile(itl_chunked, 95):>12.1f}")
# P99 = tail latency (most impacted by prefill stalls)
print(f"{'P99':<12} {np.percentile(itl_no_chunk, 99):>12.1f} {np.percentile(itl_chunked, 99):>12.1f}")
# Max = worst-case single iteration latency
print(f"{'Max':<12} {np.max(itl_no_chunk):>12.1f} {np.max(itl_chunked):>12.1f}")

In [ ]:
# ============================================================
# PLOT 1: ITL Timeline showing stall vs smooth behavior
# ============================================================
# Create two vertically stacked subplots sharing x-axis
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Top subplot: without chunking shows massive spike
# Red color indicates the problematic behavior
axes[0].plot(itl_no_chunk, color='#dc2626', linewidth=1.2, label='No Chunked Prefill')
# Add P50 reference line to show how far spike deviates
axes[0].axhline(y=np.percentile(itl_no_chunk, 50), color='#64748b',
                linestyle='--', alpha=0.5, label='P50')
# Label the y-axis with the metric being plotted
axes[0].set_ylabel('ITL (ms)')
# Title explains what the reader is seeing
axes[0].set_title('Without Chunked Prefill: Decode Stalls During Long Prefill')
# Legend in upper right to avoid overlapping data
axes[0].legend(loc='upper right')
# Set y-limit to show the full spike height
axes[0].set_ylim(0, max(itl_no_chunk) * 1.2)

# Bottom subplot: with chunking shows small controlled bumps
# Green color indicates the healthy behavior
axes[1].plot(itl_chunked, color='#16a34a', linewidth=1.2, label='Chunked Prefill')
# Same P50 reference for comparison
axes[1].axhline(y=np.percentile(itl_chunked, 50), color='#64748b',
                linestyle='--', alpha=0.5, label='P50')
# Label axes
axes[1].set_ylabel('ITL (ms)')
axes[1].set_xlabel('Iteration')
axes[1].set_title('With Chunked Prefill: Decode Continues with Small Overhead')
axes[1].legend(loc='upper right')
# Scale to 2x max to show how small the bumps are
axes[1].set_ylim(0, max(itl_chunked) * 2)

# Adjust spacing between subplots
plt.tight_layout()
# Save the figure as PNG for embedding
plt.savefig('itl_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: itl_timeline.png")

In [ ]:
# ============================================================
# PLOT 2: CDF (Cumulative Distribution Function) of ITL
# Shows tail latency behavior clearly
# ============================================================
# Create single figure for the CDF comparison
fig, ax = plt.subplots(figsize=(10, 5))

# Sort ITL values to compute CDF (sorted = percentile mapping)
sorted_no_chunk = np.sort(itl_no_chunk)
# Sort chunked values for its CDF
sorted_chunked = np.sort(itl_chunked)
# Percentile axis from 0 to 100 (one point per iteration)
cdf = np.linspace(0, 1, NUM_ITERATIONS)

# Plot CDF for no-chunking case (red = bad tail)
ax.plot(sorted_no_chunk, cdf * 100, color='#dc2626', linewidth=2, label='No Chunking')
# Plot CDF for chunked case (green = tight distribution)
ax.plot(sorted_chunked, cdf * 100, color='#16a34a', linewidth=2, label='Chunked Prefill')

# Add horizontal P99 reference line
ax.axhline(y=99, color='#64748b', linestyle=':', alpha=0.7)
# Label the P99 line for clarity
ax.text(sorted_no_chunk[-1] * 0.7, 99.5, 'P99', color='#64748b', fontsize=10)

# Label axes to explain what the chart shows
ax.set_xlabel('ITL (ms)')
ax.set_ylabel('Percentile')
ax.set_title('ITL CDF: Chunked Prefill Eliminates Tail Latency')
ax.legend(loc='lower right')
# Light grid for readability
ax.grid(True, alpha=0.3)

# Save and display
plt.tight_layout()
plt.savefig('itl_cdf.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: itl_cdf.png")

In [ ]:
# ============================================================
# PLOT 3: Token budget sensitivity analysis
# Shows the tradeoff between P99 ITL and TTFT
# ============================================================
# Different token budgets to compare (smaller = tighter ITL, higher TTFT)
chunk_budgets = [1024, 2048, 4096, 8192, 16384]
# Store P99 ITL for each budget configuration
p99_values = []
# Store TTFT (time to first token) for each budget
ttft_values = []

# Simulate each budget configuration
for budget in chunk_budgets:
    # Run chunked prefill simulation with this budget
    itl = simulate_chunked_prefill(
        NUM_ACTIVE_REQUESTS, PREFILL_TOKENS, PREFILL_TIME_PER_TOKEN_MS,
        DECODE_TIME_PER_TOKEN_MS, budget, NUM_ITERATIONS, LONG_PREFILL_ARRIVES_AT
    )
    # Record the P99 ITL achieved with this budget
    p99_values.append(np.percentile(itl, 99))

    # Calculate TTFT: how long until the new request starts decoding
    # Available prefill capacity per iteration
    chunk_size = budget - NUM_ACTIVE_REQUESTS
    # Number of iterations to complete full prefill
    num_chunks = int(np.ceil(PREFILL_TOKENS / chunk_size))
    # Time per iteration during chunked prefill
    iter_time = (NUM_ACTIVE_REQUESTS * DECODE_TIME_PER_TOKEN_MS +
                 chunk_size * PREFILL_TIME_PER_TOKEN_MS)
    # Total TTFT = chunks needed * time per chunk iteration
    ttft_values.append(num_chunks * iter_time)

# Create figure with dual y-axes for the tradeoff visualization
fig, ax1 = plt.subplots(figsize=(10, 5))

# Bar chart: P99 ITL on left axis (blue, want this low)
color1 = '#2563eb'
ax1.bar(range(len(chunk_budgets)), p99_values, color=color1, alpha=0.7, label='P99 ITL')
ax1.set_xlabel('Token Budget')
ax1.set_ylabel('P99 ITL (ms)', color=color1)
# Set x-tick labels to show actual budget values
ax1.set_xticks(range(len(chunk_budgets)))
ax1.set_xticklabels([str(b) for b in chunk_budgets])

# Line chart: TTFT on right axis (red, tradeoff cost)
ax2 = ax1.twinx()
color2 = '#dc2626'
ax2.plot(range(len(chunk_budgets)), ttft_values, color=color2, linewidth=2,
         marker='o', label='TTFT')
ax2.set_ylabel('TTFT (ms)', color=color2)

# Title explains the key insight
ax1.set_title('Chunk Size Tradeoff: Smaller Budget = Lower P99 ITL but Higher TTFT')
# Combined legend from both axes
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
# Save the tradeoff analysis chart
plt.savefig('chunk_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: chunk_tradeoff.png")

## Key Takeaways

1. **Without chunking**: a single long prefill creates a P99 ITL spike 15-20x above normal
2. **With chunking**: P99 ITL stays within 2x of P50 regardless of incoming prompt length
3. **Budget tradeoff**: smaller budgets give tighter ITL bounds but increase TTFT for new requests
4. **Sweet spot**: 4096 token budget works well for most interactive workloads (64 decode + 4032 prefill per iteration)